# Orders dataset cleaning 

In [ ]:
import pandas as pd
import numpy as np

In [ ]:


orders = pd.read_csv(r"C:\Users\MAINAK\Desktop\Data-Science\projects\MarketLens\data\raw\olist_orders_dataset.csv")

### Step 1 — Load & verify

- Shape
- Columns
- Data types
- Null values
- Duplicate rows

In [ ]:
orders.shape

In [ ]:
orders.columns

In [ ]:
orders.dtypes

In [ ]:
orders.isnull().sum()

In [ ]:
orders.duplicated().sum()

### Step 2 — Nulls vs Order Status

In [ ]:
orders[orders["order_approved_at"].isnull()]["order_status"].value_counts()

In [ ]:
orders[orders["order_delivered_carrier_date"].isnull()]["order_status"].value_counts()

In [ ]:
orders[orders["order_delivered_customer_date"].isnull()]["order_status"].value_counts()

In [ ]:
orders[
    (orders["order_approved_at"].isnull()) &
    (orders["order_status"] == "delivered")
]

In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_delivered_carrier_date"] = pd.to_datetime(orders["order_delivered_carrier_date"])

In [ ]:
orders[
    (orders["order_approved_at"].isnull()) &
    (orders["order_status"] == "delivered")
][["order_purchase_timestamp", "order_delivered_carrier_date"]]


In [ ]:
purchase_carrier_days = orders["order_delivered_carrier_date"] - orders["order_purchase_timestamp"]


In [ ]:
purchase_carrier_days.max()

### Date Columns Datetime Conversion

In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

In [ ]:
orders.dtypes

In [ ]:
invalid_delivery = orders[orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]]
len(invalid_delivery)

In [ ]:
invalid_carrier = orders[orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"]]
len(invalid_carrier)

In [ ]:
orders["has_invalid_timeline"] = (
    (orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]) |
    (orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"])
)

orders["has_invalid_timeline"].value_counts()

In [ ]:
orders["order_status"].value_counts()

In [ ]:
orders.columns

In [ ]:
import os

os.makedirs("../data/processed", exist_ok=True)
orders.to_csv("../data/processed/orders_cleaned.csv", index=False)

## Order_Items data set Cleaning

In [ ]:
order_items = pd.read_csv(r"C:\Users\MAINAK\Desktop\Data-Science\projects\MarketLens\data\raw\olist_order_items_dataset.csv")

#  shape and columns 

print("Shape : ",order_items.shape)
print("\n Columns : ",order_items.columns.tolist())


In [ ]:
#. Check missing Values 
print("\nCheck missing values : \n",order_items.isnull().sum())

# exact duplicates row Check 

print("\n Duplicates rows : \n ", order_items.duplicated().sum())

In [ ]:
# Date _times change 
order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"])

# Statistical summary of numerical columns
order_items[["price", "freight_value"]].describe()

In [ ]:
print("Prices <= 0:", (order_items["price"] <= 0).sum())
print("Freight < 0:", (order_items["freight_value"] < 0).sum())
print("Freight == 0 (Free shipping):", (order_items["freight_value"] == 0).sum())

In [ ]:
# 1. Composite key uniqueness check
composite_duplicates = order_items.duplicated(subset=["order_id", "order_item_id"]).sum()
print("Composite key duplicates (order_id + order_item_id):", composite_duplicates)

# 2. Referential integrity: Orphan items check against orders
orphan_items = (~order_items["order_id"].isin(orders["order_id"])).sum()
print("Orphan items count (orders me missing):", orphan_items)

In [ ]:
# ensure processed directory exists
os.makedirs("../data/processed", exist_ok=True)

# save cleaned order_items
order_items.to_csv("../data/processed/order_items_cleaned.csv", index=False)

## Order_payments

In [ ]:
# 1. Load dataset
payments = pd.read_csv(r"C:\Users\MAINAK\Desktop\Data-Science\projects\MarketLens\data\raw\olist_order_payments_dataset.csv")

# 2. Basic info and shape
print("Shape:", payments.shape)
print("\nColumns:", payments.columns.tolist())
payments.info()

# 3. Missing values & Duplicates check
print("\nMissing values:\n", payments.isnull().sum())
print("\nExact duplicate rows:", payments.duplicated().sum())

In [ ]:
# Payment types distribution
print("Payment Types:\n", payments["payment_type"].value_counts())

In [ ]:
# 1. Summary statistics
print(payments[["payment_installments", "payment_value"]].describe())

# 2. Potential anomalies count
print("\nZero or Negative Payments:", (payments["payment_value"] <= 0).sum())
print("Zero Installments:", (payments["payment_installments"] == 0).sum())

In [40]:
# 1. Inspect 'not_defined' payment types
print("--- NOT DEFINED ROWS ---")
display(payments[payments["payment_type"] == "not_defined"])

# 2. Inspect zero payment values
print("\n--- ZERO PAYMENT VALUE ROWS ---")
display(payments[payments["payment_value"] == 0])

# 3. Inspect zero installments
print("\n--- ZERO INSTALLMENTS ROWS ---")
display(payments[payments["payment_installments"] == 0])

--- NOT DEFINED ROWS ---


,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0



--- ZERO PAYMENT VALUE ROWS ---


,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0



--- ZERO INSTALLMENTS ROWS ---


,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [41]:
# Zero-payment order IDs ka status check
zero_pay_orders = payments[payments["payment_value"] == 0]["order_id"]
orders[orders["order_id"].isin(zero_pay_orders)][["order_id", "order_status"]]

,order_id,order_status
423,45ed6e85398a87c253db47c2d9f48216,delivered
1130,00b1cb0320190ca0daa2c88b35206009,canceled
25932,fa65dad1b0e818e3ccc5cb0e39231352,shipped
32287,6ccb433e00daae1283ccc956189c82ae,delivered
39919,4637ca194b6387e2d538dc89b124b0ee,canceled
40235,c8c528189310eaa44a745b8d9d26908b,canceled
85786,b23878b3e8eb4d25a158f57d96331b18,delivered
96970,8bcbe01d44d147f901cd3192671144db,delivered


In [42]:
# 1. Drop 3 undefined, zero-value phantom records
payments_cleaned = payments[payments["payment_type"] != "not_defined"].copy()

# 2. Impute 0 installments to 1 (business rule for credit card transactions)
payments_cleaned.loc[payments_cleaned["payment_installments"] == 0, "payment_installments"] = 1

# 3. Verification checks
print("Remaining rows:", payments_cleaned.shape[0])
print("Zero installments left:", (payments_cleaned["payment_installments"] == 0).sum())
print("Not-defined count:", (payments_cleaned["payment_type"] == "not_defined").sum())

Remaining rows: 103883
Zero installments left: 0
Not-defined count: 0


In [43]:
# Check for orphan orders in payments
orphan_payments = (~payments_cleaned["order_id"].isin(orders["order_id"])).sum()
print("Orphan payment records:", orphan_payments)

Orphan payment records: 0


In [44]:
os.makedirs("../data/processed", exist_ok=True)
payments_cleaned.to_csv("../data/processed/order_payments_cleaned.csv", index=False)
print("Saved successfully to data/processed/order_payments_cleaned.csv")

Saved successfully to data/processed/order_payments_cleaned.csv
